In [6]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup

VIEW_URL = "https://kssc.mods.go.kr:8443/ksscNew_web/kssc/common/ClassificationContentMainTreeListView.do"
MAIN_URL = "https://kssc.mods.go.kr:8443/ksscNew_web/kssc/common/ClassificationContent.do?gubun=1&strCategoryNameCode=001&categoryMenu=007&addGubun=no"


def fetch_ind_page(session, ind_code):
    data = {
        "strCategoryNameCode": "001",
        "strCategoryCode": str(ind_code).zfill(2),
        "strCategoryDegree": "11",
        "categoryMenu": "007",
        "strGubun": "0",
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": MAIN_URL,
        "Origin": "https://kssc.mods.go.kr:8443",
        "X-Requested-With": "XMLHttpRequest",
        "X-Prototype-Version": "1.6.0.3",
    }

    res = session.post(VIEW_URL, data=data, headers=headers, timeout=20)
    res.raise_for_status()
    return res.text


def extract_ind_def(html):
    soup = BeautifulSoup(html, "html.parser")

    for tr in soup.find_all("tr"):
        th = tr.find("th")
        td = tr.find("td")

        if th and td and th.get_text(strip=True) == "설명":
            return td.get_text(" ", strip=True)

    return None


def main():
    df = pd.read_csv("..\\data\\ind_basic.csv", dtype={"ind_code": str})

    session = requests.Session()
    session.get(
        MAIN_URL,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    ind_defs = []

    for ind_code in df["ind_code"]:
        code = str(ind_code).zfill(2)
        print(f"processing {code} ...")

        try:
            html = fetch_ind_page(session, code)
            ind_def = extract_ind_def(html)
            print(f"ind_def[{code}] = {ind_def}")
            ind_defs.append(ind_def)

        except Exception as e:
            print(f"failed {code}: {e}")
            ind_defs.append(None)

        time.sleep(0.5)

    df["ind_def"] = ind_defs
    df.to_csv("..\\data\\ind_basic_filled.csv", index=False, encoding="utf-8-sig")
    print("saved: ..\\data\\ind_basic_filled.csv")


if __name__ == "__main__":
    main()

processing 01 ...
ind_def[01] = 작물재배업, 축산업, 작물재배 및 축산 복합농업, 작물재배 및 축산 관련 서비스업과 수렵업 및 수렵 관련 서비스업을 포함한다.
processing 02 ...
ind_def[02] = 
processing 03 ...
ind_def[03] = 바다, 강, 호수, 하천 등에서 어류, 갑각류, 연체동물, 해조류 및 기타 수산 동·식물을 채취·포획하거나 증식 또는 양식하는 산업활동과 이에 관련된 서비스를 제공하는 산업활동을 말한다. <제외> ·고래 이외의 수산 포유동물 육지 포획( 01500 )
processing 05 ...
ind_def[05] = 무연탄, 유연탄, 갈탄 등의 석탄(토탄 제외)을 채굴하는 산업활동과 각종 형태의 유전, 천연가스전 또는 역청광물 채굴장에서 원유, 천연가스 또는 역청질 광물, 석유 함유 혈암, 타르 모래를 채굴·추출·채취하는 산업활동을 말한다.
processing 06 ...
ind_def[06] = 철 및 철 이외의 금속(비철금속)을 함유한 금속광물을 채굴하는 산업활동을 말한다. 통상적으로 금속광물 채광활동에 부수되는 광물의 파쇄, 마쇄, 자성 및 중력에 의한 분리, 선별, 체질, 부유, 분말의 응집처리(입상, 구형상, 원통상 등), 건조, 배소, 자화 또는 산화하기 위한 하소 등과 같이 기본적인 화학적 구조를 변화시키지 않는 범위 내에서 수행되는 각종 금속광물의 정광활동은 채광활동과의 결합여부를 불문하고 여기에 포함한다. 우라늄 및 토륨 채굴 활동도 포함한다.
processing 07 ...
ind_def[07] = 석탄, 석유 및 천연가스, 금속광물을 제외한 비금속광물의 채굴 또는 채취활동과 채광활동에 부수되는 파쇄, 마쇄, 절단, 세척, 건조, 분리, 혼합 등의 활동을 포함한다. 토탄 채굴 활동도 포함한다.
processing 08 ...
ind_def[08] = 
processing 10 ...
ind_def[10] = 농업, 임업 및 어업에서 생산된 산출물을 사람이나